# Step 5: Merge Features and Create Train/Val Split (with Horizon Support)
## Modified to ensure column_step_map.json is available

This notebook merges all feature tables and verifies that column_step_map.json exists for horizon model training.

In [1]:
import time
import json
from pathlib import Path
import pandas as pd
import numpy as np
start_time = time.time()
print("=" * 60)
print("Step 5: Merging Features and Creating Train/Val Split")
print("=" * 60)

Step 5: Merging Features and Creating Train/Val Split


In [2]:
# AUTO-DETECT PROJECT ROOT
import os
from pathlib import Path

def find_project_root(marker_file='CLAUDE.md', max_depth=5):
    """Find project root by looking for marker file"""
    current = Path.cwd()
    
    # Try current directory and parents
    for _ in range(max_depth):
        if (current / marker_file).exists():
            return current
        current = current.parent
    
    # If not found, return current working directory
    print(f"WARNING: Could not find {marker_file}, using current directory")
    return Path.cwd()

# Find and change to project root
project_root = find_project_root()
os.chdir(project_root)

print(f"Project root: {project_root}")
print(f"Current directory: {Path.cwd()}")
print(f"\nData directory exists: {(Path.cwd() / 'data').exists()}")
print(f"Outputs directory exists: {(Path.cwd() / 'outputs').exists()}")

Project root: d:\capstone_pipeline
Current directory: d:\capstone_pipeline

Data directory exists: True
Outputs directory exists: True


In [3]:
# Setup paths
output_dir = Path("outputs")
features_dir = output_dir / "features"
features_dir.mkdir(parents=True, exist_ok=True)

sentinel = features_dir / "05_merge.done"
force = False  # Set to True to force rebuild

if sentinel.exists() and not force:
    print(f"[OK] Features already merged (found {sentinel})")
    print(f"  Set force=True to rebuild")
else:
    print("Merging features...")

[OK] Features already merged (found outputs\features\05_merge.done)
  Set force=True to rebuild


In [4]:
# Check that all prerequisite files exist
required_files = [
    features_dir / "target.parquet",
    features_dir / "sensor_features.parquet",
    features_dir / "spc_features.parquet",
    features_dir / "lot_features.parquet"
]

for file in required_files:
    if not file.exists():
        raise FileNotFoundError(
            f"Required file not found: {file}\n"
            f"Please run previous pipeline steps first."
        )

In [5]:
# Load all feature tables
print("Loading target...")
target = pd.read_parquet(features_dir / "target.parquet")
print(f"  Target shape: {target.shape}")

print("Loading sensor features...")
sensor = pd.read_parquet(features_dir / "sensor_features.parquet")
if 'WAFER_SCRIBE' not in sensor.columns:
    sensor = sensor.reset_index()
print(f"  Sensor features shape: {sensor.shape}")

print("Loading SPC features...")
spc = pd.read_parquet(features_dir / "spc_features.parquet")
if 'WAFER_SCRIBE' not in spc.columns:
    spc = spc.reset_index()
print(f"  SPC features shape: {spc.shape}")

print("Loading lot features...")
lot = pd.read_parquet(features_dir / "lot_features.parquet")
print(f"  Lot features shape: {lot.shape}")

Loading target...
  Target shape: (16817, 4)
Loading sensor features...
  Sensor features shape: (16817, 9889)
Loading SPC features...
  SPC features shape: (18669, 408)
Loading lot features...
  Lot features shape: (16817, 9)


In [6]:
# Merge all features
print("\nMerging all features...")
df = target.copy()
print(f"  Starting with target: {df.shape}")

df = df.merge(sensor, on='WAFER_SCRIBE', how='left')
print(f"  After sensor merge: {df.shape}")

df = df.merge(spc, on='WAFER_SCRIBE', how='left')
print(f"  After SPC merge: {df.shape}")

df = df.merge(lot, on='WAFER_SCRIBE', how='left', suffixes=('', '_lot'))
print(f"  After lot merge: {df.shape}")

# Remove duplicate LOT_ID column if it exists
if 'LOT_ID_lot' in df.columns:
    df = df.drop(columns=['LOT_ID_lot'])

# Convert PARAM_END_DATETIME to datetime
df['PARAM_END_DATETIME'] = pd.to_datetime(df['PARAM_END_DATETIME'])

# Sort by datetime
df = df.sort_values('PARAM_END_DATETIME')


Merging all features...
  Starting with target: (16817, 4)
  After sensor merge: (16817, 9892)
  After SPC merge: (16817, 10299)
  After lot merge: (16817, 10307)


In [7]:
# Temporal split by lot (80/20)
print("\nCreating temporal train/val split...")

# Get unique lots with their earliest timestamp
lot_times = df.groupby('LOT_ID')['PARAM_END_DATETIME'].min().reset_index()
lot_times = lot_times.sort_values('PARAM_END_DATETIME')

# Split lots 80/20
n_lots = len(lot_times)
train_cutoff_idx = int(0.8 * n_lots)
train_lots = lot_times.iloc[:train_cutoff_idx]['LOT_ID'].tolist()
val_lots = lot_times.iloc[train_cutoff_idx:]['LOT_ID'].tolist()

print(f"  Total lots: {n_lots}")
print(f"  Train lots: {len(train_lots)} (80%)")
print(f"  Val lots: {len(val_lots)} (20%)")

# Create train/val splits
train_df = df[df['LOT_ID'].isin(train_lots)].copy()
val_df = df[df['LOT_ID'].isin(val_lots)].copy()

print(f"\n  Train size: {len(train_df)} wafers")
print(f"  Val size: {len(val_df)} wafers")

# Print class balance
print(f"\nClass balance:")
print(f"  Train outliers: {train_df['is_outlier'].sum()} "
      f"({100 * train_df['is_outlier'].mean():.2f}%)")
print(f"  Val outliers: {val_df['is_outlier'].sum()} "
      f"({100 * val_df['is_outlier'].mean():.2f}%)")


Creating temporal train/val split...
  Total lots: 801
  Train lots: 640 (80%)
  Val lots: 161 (20%)

  Train size: 13510 wafers
  Val size: 3307 wafers

Class balance:
  Train outliers: 234 (1.73%)
  Val outliers: 52 (1.57%)


In [8]:
# Identify feature columns (exclude metadata)
metadata_cols = ['WAFER_SCRIBE', 'LOT_ID', 'is_outlier', 'PARAM_END_DATETIME', 
                'first_start_time']
feature_cols = [col for col in df.columns if col not in metadata_cols]

print(f"\nTotal feature columns: {len(feature_cols)}")


Total feature columns: 10302


In [9]:
# Identify categorical columns for CatBoost
cat_cols = []
for col in feature_cols:
    is_categorical = (
        df[col].dtype == 'object' or
        df[col].dtype.name == 'string' or
        str(df[col].dtype).startswith('str') or
        col.startswith('LOT_ID') or
        '__EQUIP' in col or
        '__POSITION' in col or
        col in ['first_step', 'last_step']
    )
    if is_categorical and col in df.columns:
        cat_cols.append(col)
        # Fill NaN in categorical columns with 'UNKNOWN' for CatBoost
        df[col] = df[col].fillna('UNKNOWN').astype(str)

print(f"Categorical feature columns: {len(cat_cols)}")

# Get indices of categorical columns in feature list
cat_feature_indices = [i for i, col in enumerate(feature_cols) if col in cat_cols]

Categorical feature columns: 48


In [10]:
# Save datasets
print("\nSaving datasets...")
train_df.to_parquet(features_dir / "train.parquet", index=False)
val_df.to_parquet(features_dir / "val.parquet", index=False)

# Save feature metadata
with open(features_dir / "feature_columns.json", 'w') as f:
    json.dump(feature_cols, f, indent=2)

with open(features_dir / "cat_feature_indices.json", 'w') as f:
    json.dump(cat_feature_indices, f, indent=2)

print(f"  Saved train.parquet")
print(f"  Saved val.parquet")
print(f"  Saved feature_columns.json")
print(f"  Saved cat_feature_indices.json")


Saving datasets...
  Saved train.parquet
  Saved val.parquet
  Saved feature_columns.json
  Saved cat_feature_indices.json


## NEW: Verify Column-Step Mapping Exists

Ensure column_step_map.json exists and is complete for horizon model training.

In [11]:
# Verify column_step_map.json exists
print("\n" + "=" * 60)
print("Verifying column-step mapping for horizon models...")
print("=" * 60)

column_map_file = features_dir / "column_step_map.json"

if column_map_file.exists():
    with open(column_map_file, 'r') as f:
        column_step_map = json.load(f)
    
    print(f"\n[OK] column_step_map.json exists with {len(column_step_map)} mappings")
    
    # Check coverage
    mapped_features = set(column_step_map.keys())
    all_features = set(feature_cols)
    
    # Lot features should have SeqNo=0 (always included)
    lot_feature_cols = [c for c in feature_cols if c.startswith('lot_') or 
                       c.startswith('first_') or c.startswith('last_') or 
                       c == 'step_coverage']
    
    # Add missing lot features to map with SeqNo=0
    missing_features = all_features - mapped_features
    if missing_features:
        print(f"\n  Adding {len(missing_features)} unmapped features with SeqNo=0 "
              f"(lot features and metadata)...")
        for feat in missing_features:
            column_step_map[feat] = 0
        
        # Save updated map
        with open(column_map_file, 'w') as f:
            json.dump(column_step_map, f, indent=2)
        print(f"  Updated column_step_map.json to {len(column_step_map)} total mappings")
    
    # Show statistics
    seqno_values = list(column_step_map.values())
    print(f"\n  SeqNo range: {min(seqno_values)} to {max(seqno_values)}")
    print(f"  Features with SeqNo=0 (always included): "
          f"{sum(1 for v in seqno_values if v == 0)}")
    print(f"  Features mapped to process steps: "
          f"{sum(1 for v in seqno_values if v > 0)}")
    
else:
    print(f"\n  WARNING: column_step_map.json not found at {column_map_file}")
    print(f"  Horizon models will not work without this file.")
    print(f"  Please run steps 02 and 03 with the updated versions.")


Verifying column-step mapping for horizon models...

[OK] column_step_map.json exists with 10295 mappings

  Adding 7 unmapped features with SeqNo=0 (lot features and metadata)...
  Updated column_step_map.json to 10302 total mappings

  SeqNo range: 0 to 240
  Features with SeqNo=0 (always included): 53
  Features mapped to process steps: 10249


In [12]:
# Mark complete
sentinel.touch()

elapsed = time.time() - start_time
print(f"\n[OK] Feature merge complete in {elapsed:.2f}s")
print("=" * 60)


[OK] Feature merge complete in 37.47s
